In [ ]:
import sys
import os
from pathlib import Path  # noqa: F401

import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from src.analysis.isc import (  # noqa: E402
    FREQUENCY_BANDS,
    compute_loo_isc,
    compute_loo_isc_spearman,
    compute_pairwise_isc,
    compute_pairwise_isc_spearman,
    compute_sliding_window_isc,
    compute_sliding_window_isc_spearman,
)
from src.definitions.constants import ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    ExclusionCategories,
    MusicTypeVariants,
)
from src.visualization.isc_plots import (  # noqa: E402
    plot_band_overlap,
    print_data_overview,
)
from scripts.analysis_common import (  # noqa: E402
    BAND_ISC_THRESHOLDS,
    analyzers_to_datasets,
    load_analyzers,
)

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)

# EEG Inter-Subject Correlation Analysis — Per Frequency Band

This notebook computes **inter-subject correlation (ISC)** on the raw EEG
signal band-pass filtered to the five standard EEG frequency bands
(delta, theta, alpha, beta, gamma):

1. **Per-band LOO-ISC distributions** — for each band, Pearson and Spearman
   correlation against the mean of all others; histogram and per-subject violin plot
2. **Per-band pairwise ISC matrix** — symmetric subject × subject correlation
   matrix per band; per-subject mean off-diagonal for outlier detection
3. **Per-band sliding-window ISC** — time-resolved LOO-ISC per band at three
   temporal scales; Pearson vs Spearman comparison at medium window
4. **Band-overlap analysis** — raster showing which bands simultaneously
   exceed their significance thresholds

Computation functions are imported from `src.analysis.isc`.

## Configuration

In [ ]:
# ── Experiment configuration ───────────────────────────────────────────────
CONDITION = ConditionVariants.PLACEBO
MUSIC_TYPES = [MusicTypeVariants.CLASSICAL, MusicTypeVariants.PSYTRANCE]
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]

# ── Data processing flag ───────────────────────────────────────────────────
process_and_save_data = False

# ── LOO-ISC distribution parameters ───────────────────────────────────────
PLOT_PCT = 99  # clip histogram at this percentile to suppress outliers

# ── Sliding-window ISC parameters ─────────────────────────────────────────
# Three non-overlapping window sizes (same as broadband analysis).
WINDOW_FINE_SEC = 1.0     # fine / one-step window (seconds)
WINDOW_MED_SEC = 5.0      # medium window (seconds)
WINDOW_LARGE_SEC = 15.0   # large window (seconds)

# Per-band ISC significance thresholds
BAND_THRESHOLDS = BAND_ISC_THRESHOLDS

# Sub-sample channels for Spearman to keep runtime manageable during exploration.
# Set to None to use all channels (much slower).
N_CH_SUBSAMPLE = 64

# ── Plot saving ────────────────────────────────────────────────────────────
SAVE_PLOTS = True

PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR / "02-isc-broadband-analysis" / "plots" / "bands"
)
print(f"Plots will be saved to: {PLOTS_DIR}")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

## Data Loading

In [ ]:
# Load (or process-and-save) one EEGSummarizedAnalyzer per music type.
# normalize_data=False: keep raw amplitudes for this ISC analysis.
analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    process_and_save_data,
    normalize_data=False,
)
datasets = analyzers_to_datasets(analyzers)
print_data_overview(datasets)

## Dataset Selection

Change `LABEL` to switch between music types. All analysis cells below use
`ad`, `data`, `n_subjects`, `n_channels`, `n_times`, and `sfreq`.

In [ ]:
LABEL = MusicTypeVariants.CLASSICAL.value
# LABEL = MusicTypeVariants.PSYTRANCE.value

ad = datasets[LABEL]
data = ad.data        # (n_subjects, n_channels, n_times) — unnormalized
sfreq = ad.sfreq

n_subjects, n_channels, n_times = data.shape
time = np.arange(n_times) / sfreq   # seconds

print(f"Dataset : {LABEL}")
print(f"Shape   : {data.shape}  (subjects × channels × time points)")
print(f"Duration: {n_times / sfreq:.1f} s  @  {sfreq} Hz")
print("Data: unnormalized (raw amplitude).")

# Available frequency bands
print("\nFrequency bands:")
for band, (lo, hi) in FREQUENCY_BANDS.items():
    thr = BAND_THRESHOLDS.get(band, 0.035)
    print(f"  {band:6s}: {lo}–{hi} Hz  (threshold r = {thr})")

## Section 1 — Per-Band LOO-ISC Distribution (Pearson vs. Spearman)

For each frequency band, the raw EEG is band-pass filtered and leave-one-out
ISC is computed per channel for each subject.

For each band:
- **Histogram** — distribution of channel-wise mean LOO-ISC (Pearson vs. Spearman)
- **Per-subject violin + strip** — each dot is one subject's mean band-LOO-ISC
  averaged over channels; reveals band-specific outlier subjects

In [ ]:
# Channel subsampling for Spearman (same seed as broadband for reproducibility)
if N_CH_SUBSAMPLE is not None and N_CH_SUBSAMPLE < n_channels:
    rng = np.random.default_rng(42)
    ch_idx = np.sort(rng.choice(n_channels, N_CH_SUBSAMPLE, replace=False))
    print(f"Using {N_CH_SUBSAMPLE}/{n_channels} randomly subsampled channels for Spearman.")
else:
    ch_idx = np.arange(n_channels)

band_iscs: dict[str, tuple] = {}         # (loo_pearson, mean_pearson)
band_iscs_spearman: dict[str, tuple] = {}  # (loo_spearman, mean_spearman) on subsampled channels

BAND_COLORS = {
    "delta": "#4e79a7",
    "theta": "#f28e2b",
    "alpha": "#59a14f",
    "beta": "#e15759",
    "gamma": "#b07aa1",
}

for band, (l_freq, h_freq) in FREQUENCY_BANDS.items():
    print(f"  {band:6s} ({l_freq}–{h_freq} Hz) — Pearson LOO-ISC …")
    filtered = ad.filter_to_band(l_freq, h_freq)
    loo, mean_isc = compute_loo_isc(filtered.data)
    band_iscs[band] = (loo, mean_isc)

    print(f"  {band:6s} ({l_freq}–{h_freq} Hz) — Spearman LOO-ISC …")
    loo_sp, mean_isc_sp = compute_loo_isc_spearman(filtered.data[:, ch_idx, :])
    band_iscs_spearman[band] = (loo_sp, mean_isc_sp)

for band, (l_freq, h_freq) in FREQUENCY_BANDS.items():
    loo_p, mean_p = band_iscs[band]
    loo_s, mean_s = band_iscs_spearman[band]
    n_ch_p = mean_p.shape[0]
    color = BAND_COLORS.get(band, "steelblue")
    thr = BAND_THRESHOLDS.get(band, 0.035)

    # ── Histogram: Pearson vs. Spearman side by side ────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    for ax, mean_isc, col, method in zip(
        axes,
        [mean_p, mean_s],
        [color, "darkorange"],
        ["Pearson", "Spearman"],
    ):
        clip = np.percentile(mean_isc, PLOT_PCT)
        n_out = int((mean_isc > clip).sum())
        vals = mean_isc[mean_isc <= clip]
        counts, edges = np.histogram(vals, bins=30)
        pct_vals = counts / len(mean_isc) * 100
        centers = 0.5 * (edges[:-1] + edges[1:])
        _hist_colors = [col if c >= 0 else "#e57373" for c in centers]
        ax.bar(
            centers, pct_vals, width=(edges[1] - edges[0]) * 0.9,
            color=_hist_colors, alpha=0.75, edgecolor="none",
        )
        ax.axvline(np.mean(mean_isc), color=".2", ls="--", lw=1.3,
                   label=f"Mean = {np.mean(mean_isc):.4f}")
        ax.axvline(np.median(mean_isc), color="crimson", ls=":", lw=1.2,
                   label=f"Median = {np.median(mean_isc):.4f}")
        ax.axvline(0, color="gray", ls="-", lw=0.8, alpha=0.5, label="r = 0")
        ax.axvline(thr, color="tomato", ls="--", lw=0.9, alpha=0.6,
                   label=f"Threshold = {thr}")
        ax.set_xlabel("LOO-ISC (r)")
        ax.set_ylabel("% of channels")
        ax.set_title(
            f"[{LABEL} / {band.upper()}]  {method} LOO-ISC\n"
            f"({n_out} channel(s) clipped at {PLOT_PCT}th pct)"
        )
        ax.legend(frameon=False, fontsize=9)
    sns.despine(fig=fig)
    fig.suptitle(
        f"{LABEL} / {band.upper()} — Channel-wise mean LOO-ISC distribution",
        fontsize=13, y=1.02,
    )
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(
            PLOTS_DIR / f"loo_isc_distribution_{band}_{LABEL}.png",
            dpi=150, bbox_inches="tight",
        )
    plt.show()

    # ── Per-subject violin + strip ────────────────────────────────────────
    df_subj = pd.concat([
        pd.DataFrame({
            "subject": np.arange(n_subjects),
            "mean_isc": loo_p.mean(axis=1),
            "method": "Pearson",
        }),
        pd.DataFrame({
            "subject": np.arange(n_subjects),
            "mean_isc": loo_s.mean(axis=1),
            "method": "Spearman",
        }),
    ])
    fig2, ax2 = plt.subplots(figsize=(7, 4))
    sns.violinplot(
        data=df_subj, x="method", y="mean_isc",
        palette={"Pearson": color, "Spearman": "darkorange"},
        inner="box", ax=ax2,
    )
    sns.stripplot(
        data=df_subj, x="method", y="mean_isc",
        color=".25", size=5, jitter=True, alpha=0.65, ax=ax2,
    )
    ax2.axhline(0, color="gray", ls="--", lw=0.8, label="r = 0")
    ax2.set_xlabel("Correlation method")
    ax2.set_ylabel("Mean LOO-ISC across channels (r)")
    ax2.set_title(f"[{LABEL} / {band.upper()}]  Per-subject mean LOO-ISC")
    ax2.legend(frameon=False, fontsize=9)
    sns.despine(fig=fig2)
    fig2.tight_layout()
    if SAVE_PLOTS:
        fig2.savefig(
            PLOTS_DIR / f"loo_isc_per_subject_{band}_{LABEL}.png",
            dpi=150, bbox_inches="tight",
        )
    plt.show()

print("\nSummary:")
for band in FREQUENCY_BANDS:
    mean_p = band_iscs[band][1]
    mean_s = band_iscs_spearman[band][1]
    print(
        f"  {band:6s}  Pearson mean={mean_p.mean():.4f}  "
        f"Spearman mean={mean_s.mean():.4f}"
    )

## Section 2 — Per-Band Pairwise ISC Matrix (Pearson vs. Spearman)

For each frequency band, every pair of subjects is correlated (mean over channels):

- **Heatmaps** — diagonal masked so the colour scale focuses on off-diagonal spread
- **Per-subject mean off-diagonal bar chart** — subjects with consistently low values
  are candidates for quality review
- **Off-diagonal distribution** — histogram comparing Pearson vs. Spearman spread

In [ ]:
band_pair_pearson: dict[str, np.ndarray] = {}
band_pair_spearman: dict[str, np.ndarray] = {}

for band, (l_freq, h_freq) in FREQUENCY_BANDS.items():
    print(f"  {band:6s} — Pearson pairwise ISC …")
    filtered = ad.filter_to_band(l_freq, h_freq)
    band_pair_pearson[band] = compute_pairwise_isc(filtered.data)

    print(f"  {band:6s} — Spearman pairwise ISC …")
    band_pair_spearman[band] = compute_pairwise_isc_spearman(filtered.data[:, ch_idx, :])

subj_labels = [f"S{i + 1:02d}" for i in range(n_subjects)]
off_diag_mask = ~np.eye(n_subjects, dtype=bool)

for band in FREQUENCY_BANDS:
    pair_p = band_pair_pearson[band]
    pair_s = band_pair_spearman[band]
    color = BAND_COLORS.get(band, "steelblue")

    # ── Side-by-side heatmaps ─────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, mat, method in zip(
        axes,
        [pair_p, pair_s],
        ["Pearson", "Spearman"],
    ):
        mat_display = mat.copy()
        np.fill_diagonal(mat_display, np.nan)
        vmax = max(np.nanmax(np.abs(mat_display)), 0.01)
        im = ax.imshow(mat_display, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
        plt.colorbar(im, ax=ax, label="ISC (r)", shrink=0.82)
        ax.set_xticks(range(n_subjects))
        ax.set_yticks(range(n_subjects))
        ax.set_xticklabels(subj_labels, rotation=90, fontsize=7)
        ax.set_yticklabels(subj_labels, fontsize=7)
        mean_off = float(mat[off_diag_mask].mean())
        ax.set_title(
            f"[{LABEL} / {band.upper()}]  {method} pairwise ISC\n"
            f"mean off-diagonal r = {mean_off:.4f}"
        )
    sns.despine(fig=fig, left=True, bottom=True)
    fig.suptitle(f"{LABEL} / {band.upper()} — Pairwise ISC matrices", fontsize=13)
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(
            PLOTS_DIR / f"pairwise_isc_matrix_{band}_{LABEL}.png",
            dpi=150, bbox_inches="tight",
        )
    plt.show()

    # ── Per-subject mean off-diagonal ISC ─────────────────────────────
    mean_off_p_subj = np.array([pair_p[i, off_diag_mask[i]].mean() for i in range(n_subjects)])
    mean_off_s_subj = np.array([pair_s[i, off_diag_mask[i]].mean() for i in range(n_subjects)])
    df_off = pd.DataFrame({
        "subject": subj_labels * 2,
        "mean_pairwise_isc": np.concatenate([mean_off_p_subj, mean_off_s_subj]),
        "method": ["Pearson"] * n_subjects + ["Spearman"] * n_subjects,
    })
    fig2, ax2 = plt.subplots(figsize=(max(8, n_subjects * 0.55), 4))
    sns.barplot(
        data=df_off, x="subject", y="mean_pairwise_isc", hue="method",
        palette={"Pearson": color, "Spearman": "darkorange"},
        ax=ax2,
    )
    ax2.axhline(0, color="gray", ls="--", lw=0.8)
    ax2.set_xlabel("Subject")
    ax2.set_ylabel("Mean pairwise ISC (r)")
    ax2.set_title(
        f"[{LABEL} / {band.upper()}]  Per-subject mean pairwise ISC — low = outlier candidate"
    )
    ax2.legend(frameon=False, fontsize=9, title="Method")
    sns.despine(fig=fig2)
    fig2.tight_layout()
    if SAVE_PLOTS:
        fig2.savefig(
            PLOTS_DIR / f"pairwise_isc_per_subject_{band}_{LABEL}.png",
            dpi=150, bbox_inches="tight",
        )
    plt.show()

    # ── Off-diagonal distribution ─────────────────────────────────────
    off_p = pair_p[off_diag_mask]
    off_s = pair_s[off_diag_mask]
    n_pairs = n_subjects * (n_subjects - 1) // 2
    fig3, ax3 = plt.subplots(figsize=(8, 4))
    ax3.hist(off_p, bins=20, alpha=0.55, color=color,
             label=f"Pearson  (mean = {off_p.mean():.4f})")
    ax3.hist(off_s, bins=20, alpha=0.55, color="darkorange",
             label=f"Spearman (mean = {off_s.mean():.4f})")
    ax3.axvline(off_p.mean(), color=color, ls="--", lw=1.2)
    ax3.axvline(off_s.mean(), color="darkorange", ls="--", lw=1.2)
    ax3.axvline(0, color="gray", ls="-", lw=0.8, alpha=0.5, label="r = 0")
    ax3.set_xlabel("Pairwise ISC (r)")
    ax3.set_ylabel("Number of subject pairs")
    ax3.set_title(
        f"[{LABEL} / {band.upper()}]  Off-diagonal pairwise ISC distribution ({n_pairs} pairs)"
    )
    ax3.legend(frameon=False)
    sns.despine(fig=fig3)
    fig3.tight_layout()
    if SAVE_PLOTS:
        fig3.savefig(
            PLOTS_DIR / f"pairwise_isc_distribution_{band}_{LABEL}.png",
            dpi=150, bbox_inches="tight",
        )
    plt.show()

## Section 3 — Per-Band Sliding-Window ISC (Multi-Scale)

Time-resolved LOO-ISC per frequency band at three temporal scales.

For each band:
- **Figure A** — Bar chart (medium window, Pearson): per-window mean ISC
- **Figure B** — 3-scale stair-step overlay + per-channel heatmap (Pearson)
- **Figure C** — Pearson vs. Spearman comparison at medium window

In [ ]:
def _make_step_arr(mean_tc: np.ndarray, win_samples: int, n_times_full: int) -> np.ndarray:
    """Repeat each window value for win_samples ticks, then pad/trim to n_times_full."""
    repeated = np.repeat(mean_tc, win_samples)
    pad = n_times_full - len(repeated)
    if pad > 0:
        return np.pad(repeated, (0, pad), mode="edge")
    return repeated[:n_times_full]


C_NEG = "#e57373"
C_THRESH = "tomato"
C_SYNC = "gold"

win_configs = [
    ("fine",  WINDOW_FINE_SEC,  "lightsteelblue", "solid",  1.0),
    ("med",   WINDOW_MED_SEC,   "steelblue",      "solid",  2.2),
    ("large", WINDOW_LARGE_SEC, "saddlebrown",    "dashed", 2.5),
]

# Channel subsampling (same subset as Section 1)
data_sw = data[:, ch_idx, :]
n_ch_sw = data_sw.shape[1]
n_times_sw = data_sw.shape[2]
time_s = np.arange(n_times_sw) / sfreq
ds = max(1, n_times_sw // 8000)
t_ds = time_s[::ds]

for band, (l_freq, h_freq) in FREQUENCY_BANDS.items():
    color_band = BAND_COLORS.get(band, "steelblue")
    thr_band = BAND_THRESHOLDS.get(band, 0.035)
    filtered_sw = ad.filter_to_band(l_freq, h_freq)
    fdata_sw = filtered_sw.data[:, ch_idx, :]

    sw_data_band = {}
    for name, win_sec, color, ls, lw in win_configs:
        print(f"  {band:6s} Pearson ISC ({win_sec:.0f} s non-overlapping) …")
        isc_tc, sw_times = compute_sliding_window_isc(fdata_sw, win_sec, win_sec, sfreq)
        win_samples = int(round(win_sec * sfreq))
        mean_tc = isc_tc.mean(axis=1)
        step_arr = _make_step_arr(mean_tc, win_samples, n_times_sw)
        sw_data_band[name] = dict(
            win_sec=win_sec, isc_tc=isc_tc, sw_times=sw_times,
            mean_tc=mean_tc, step_arr=step_arr, win_samples=win_samples,
            color=color, ls=ls, lw=lw,
        )

    print(f"  {band:6s} Spearman ISC ({WINDOW_MED_SEC:.0f} s non-overlapping) …")
    sw_spear_tc, _ = compute_sliding_window_isc_spearman(
        fdata_sw, WINDOW_MED_SEC, WINDOW_MED_SEC, sfreq
    )
    sw_spear_mean = sw_spear_tc.mean(axis=1)
    sw_spear_step = _make_step_arr(
        sw_spear_mean, sw_data_band["med"]["win_samples"], n_times_sw
    )

    med = sw_data_band["med"]
    sig_mask_med = med["mean_tc"] > thr_band
    win_centers_s = med["sw_times"]
    std_per_win = med["isc_tc"].std(axis=1)
    bar_w = WINDOW_MED_SEC * 0.82

    # ── Figure A — Bar chart (medium window, Pearson) ──────────────────
    bar_colors = [
        "#5cb85c" if v > thr_band else (C_NEG if v < 0 else "steelblue")
        for v in med["mean_tc"]
    ]
    fig_bar, ax_bar = plt.subplots(figsize=(max(14, len(win_centers_s) * 0.28), 4))
    ax_bar.bar(win_centers_s, med["mean_tc"], width=bar_w, color=bar_colors, alpha=0.85, edgecolor="none")
    ax_bar.errorbar(win_centers_s, med["mean_tc"], yerr=std_per_win,
                    fmt="none", color=".35", capsize=2, linewidth=0.9, label="±std across channels")
    ax_bar.axhline(thr_band, color=C_THRESH, ls="--", lw=1.2,
                   label=f"Threshold r = {thr_band}")
    ax_bar.axhline(float(med["mean_tc"].mean()), color=".2", ls=":", lw=1.1,
                   label=f"Grand mean r = {med['mean_tc'].mean():.4f}")
    ax_bar.axhline(0, color="gray", ls="-", lw=0.6, alpha=0.4)
    bar_patches = [
        Patch(color="#5cb85c", alpha=0.85, label=f"Sig. positive (r > {thr_band})"),
        Patch(color="steelblue", alpha=0.85, label="Positive (r \u2265 0)"),
        Patch(color=C_NEG, alpha=0.85, label="Negative (r < 0)"),
    ]
    extra_h, _ = ax_bar.get_legend_handles_labels()
    ax_bar.legend(
        handles=bar_patches + extra_h,
        labels=[p.get_label() for p in bar_patches] + [h.get_label() for h in extra_h],
        frameon=False, fontsize=9,
    )
    ax_bar.set_xlabel("Time (s)")
    ax_bar.set_ylabel("Mean LOO-ISC (r)")
    ax_bar.set_title(
        f"[{LABEL} / {band.upper()}]  Per-window mean LOO-ISC  (Pearson, window = {WINDOW_MED_SEC:.0f} s)"
    )
    sns.despine(fig=fig_bar)
    fig_bar.tight_layout()
    if SAVE_PLOTS:
        fig_bar.savefig(
            PLOTS_DIR / f"sw_isc_bar_{band}_{LABEL}.png", dpi=150, bbox_inches="tight"
        )
    plt.show()

    # ── Figure B — 3-scale stair-step + heatmap ───────────────────────
    fig_ov = plt.figure(figsize=(16, 10))
    gs_ov = gridspec.GridSpec(
        2, 2, height_ratios=[2, 3], width_ratios=[30, 1],
        hspace=0.10, wspace=0.04, figure=fig_ov,
    )
    ax_isc = fig_ov.add_subplot(gs_ov[0, 0])
    ax_hm  = fig_ov.add_subplot(gs_ov[1, 0], sharex=ax_isc)
    cax    = fig_ov.add_subplot(gs_ov[1, 1])

    first_span = True
    for w in np.where(sig_mask_med)[0]:
        t_start_w = time_s[w * med["win_samples"]]
        t_end_w   = time_s[min((w + 1) * med["win_samples"] - 1, n_times_sw - 1)]
        for _ax in (ax_isc, ax_hm):
            _ax.axvspan(
                t_start_w, t_end_w, color=C_SYNC, alpha=0.22,
                label="Sig. window (med)" if (first_span and _ax is ax_isc) else "_nolegend_",
            )
        first_span = False

    label_map = {"fine": "Fine", "med": "Medium", "large": "Large"}
    for name in ("fine", "med", "large"):
        r = sw_data_band[name]
        ax_isc.step(t_ds, r["step_arr"][::ds], where="post",
                    color=r["color"], lw=r["lw"], ls=r["ls"],
                    label=f"{label_map[name]} ({r['win_sec']:.0f} s step)")

    _med_step_ds = med["step_arr"][::ds]
    ax_isc.fill_between(t_ds, 0, np.clip(_med_step_ds, 0, None),
                        step="post", alpha=0.15, color="steelblue", label="_nolegend_")
    ax_isc.fill_between(t_ds, np.clip(_med_step_ds, None, 0), 0,
                        step="post", alpha=0.25, color=C_NEG, label="_nolegend_")
    ax_isc.axhline(thr_band, color=C_THRESH, ls="--", lw=1.1,
                   label=f"Threshold r = {thr_band}")
    ax_isc.axhline(0, color="gray", ls="-", lw=0.6, alpha=0.4)
    ax_isc.set_ylabel("Mean LOO-ISC (r)")
    ax_isc.set_title(
        f"[{LABEL} / {band.upper()}]  Time-resolved LOO-ISC — 3-scale stair-step + heatmap  (Pearson)"
    )
    ax_isc.legend(frameon=False, fontsize=9)
    plt.setp(ax_isc.get_xticklabels(), visible=False)

    fine_isc   = sw_data_band["fine"]["isc_tc"]
    fine_times = sw_data_band["fine"]["sw_times"]
    _dt_fine   = fine_times[1] - fine_times[0] if len(fine_times) > 1 else WINDOW_FINE_SEC
    _t_edges   = np.r_[fine_times - _dt_fine / 2, fine_times[-1] + _dt_fine / 2]
    _ch_edges  = np.arange(n_ch_sw + 1)
    vmax_hm    = max(float(np.nanpercentile(np.abs(fine_isc), 98)), 1e-6)
    pcm = ax_hm.pcolormesh(
        _t_edges, _ch_edges, fine_isc.T, cmap="RdBu_r",
        vmin=-vmax_hm, vmax=vmax_hm, rasterized=True, shading="flat",
    )
    ax_hm.set_ylim(n_ch_sw, 0)
    fig_ov.colorbar(pcm, cax=cax, label="LOO-ISC (r)")
    ax_hm.set_xlabel("Time (s)")
    ax_hm.set_ylabel(
        f"Channel index (subsampled {n_ch_sw})" if N_CH_SUBSAMPLE else "Channel index"
    )
    sns.despine(fig=fig_ov, left=False, bottom=False)
    fig_ov.tight_layout()
    if SAVE_PLOTS:
        fig_ov.savefig(
            PLOTS_DIR / f"sw_isc_overlay_{band}_{LABEL}.png", dpi=150, bbox_inches="tight"
        )
    plt.show()

    # ── Figure C — Pearson vs. Spearman comparison ───────────────────
    fig_cmp, ax_cmp = plt.subplots(figsize=(14, 4))
    ax_cmp.step(t_ds, med["step_arr"][::ds], where="post",
                color="steelblue", lw=2.0, label=f"Pearson  ({WINDOW_MED_SEC:.0f} s step)")
    ax_cmp.step(t_ds, sw_spear_step[::ds], where="post",
                color="darkorange", lw=2.0, ls="--", label=f"Spearman ({WINDOW_MED_SEC:.0f} s step)")
    ax_cmp.fill_between(t_ds, med["step_arr"][::ds], sw_spear_step[::ds],
                        alpha=0.12, color="gray", step="post", label="Pearson − Spearman gap")
    _pearson_ds = med["step_arr"][::ds]
    _spear_ds   = sw_spear_step[::ds]
    ax_cmp.fill_between(t_ds, 0, _pearson_ds, where=_pearson_ds >= 0,
                        step="post", alpha=0.10, color="steelblue")
    ax_cmp.fill_between(t_ds, 0, _pearson_ds, where=_pearson_ds < 0,
                        step="post", alpha=0.18, color=C_NEG)
    ax_cmp.fill_between(t_ds, 0, _spear_ds, where=_spear_ds >= 0,
                        step="post", alpha=0.08, color="darkorange")
    ax_cmp.fill_between(t_ds, 0, _spear_ds, where=_spear_ds < 0,
                        step="post", alpha=0.14, color=C_NEG)
    ax_cmp.axhline(thr_band, color=C_THRESH, ls="--", lw=1.0,
                   label=f"Threshold r = {thr_band}")
    ax_cmp.axhline(0, color="gray", ls="-", lw=0.6, alpha=0.4)
    ax_cmp.set_xlabel("Time (s)")
    ax_cmp.set_ylabel("Mean LOO-ISC (r)")
    ax_cmp.set_title(
        f"[{LABEL} / {band.upper()}]  Pearson vs. Spearman  ({WINDOW_MED_SEC:.0f} s non-overlapping windows)"
    )
    ax_cmp.legend(frameon=False, fontsize=9)
    sns.despine(fig=fig_cmp)
    fig_cmp.tight_layout()
    if SAVE_PLOTS:
        fig_cmp.savefig(
            PLOTS_DIR / f"sw_isc_pearson_vs_spearman_{band}_{LABEL}.png",
            dpi=150, bbox_inches="tight",
        )
    plt.show()

    n_len = min(len(med["mean_tc"]), len(sw_spear_mean))
    corr_between = float(np.corrcoef(med["mean_tc"][:n_len], sw_spear_mean[:n_len])[0, 1])
    print(
        f"  [{band.upper()}] Correlation Pearson vs. Spearman LOO-ISC time courses: r = {corr_between:.4f}"
    )

## Section 4 — Band-Overlap Analysis

A raster plot showing, for each time window, which frequency bands simultaneously
exceed their respective ISC significance thresholds.

Windows where multiple bands show significant synchrony may indicate periods of
especially strong neural alignment across participants.

In [ ]:
# Broadband sliding-window ISC (required for the overlap raster)
print("Computing broadband sliding-window ISC for band-overlap raster …")
sw_isc_bb, sw_times_bb = compute_sliding_window_isc(
    data,
    window_sec=WINDOW_MED_SEC,
    step_sec=WINDOW_MED_SEC,
    sfreq=sfreq,
)

# Build band_sw dict using medium-window sliding-window results
band_sw: dict[str, tuple] = {}
for band, (l_freq, h_freq) in FREQUENCY_BANDS.items():
    filtered = ad.filter_to_band(l_freq, h_freq)
    tc, times = compute_sliding_window_isc(
        filtered.data,
        window_sec=WINDOW_MED_SEC,
        step_sec=WINDOW_MED_SEC,
        sfreq=sfreq,
    )
    band_sw[band] = (tc, times)

fig_overlap = plot_band_overlap(
    {LABEL: band_sw},
    bands=FREQUENCY_BANDS,
    band_thresholds=BAND_THRESHOLDS,
    broadband_sw={LABEL: (sw_isc_bb, sw_times_bb)},
    broadband_threshold=0.035,
    save_path=PLOTS_DIR / f"band_overlap_{LABEL}.png" if SAVE_PLOTS else None,
)